In [1]:
import os
import glob
import math
import onnx
import onnxoptimizer
import numpy as np
from onnx import helper, TensorProto, numpy_helper
import copy

SUBMISSION_DIR = '/Users/sweeden/kaggle/input/neurogolf-2026-agent-trace-framework-v2'
optimized_dir = os.path.join(SUBMISSION_DIR, 'optimized')
os.makedirs(optimized_dir, exist_ok=True)

GENERIC_PASSES = [
    "eliminate_deadend",
    "eliminate_identity",
    "eliminate_unused_initializer",
    "eliminate_nop_cast",
    "fuse_consecutive_squeezes",
    "fuse_consecutive_transposes",
    "fuse_bn_into_conv",
    "fuse_matmul_add_bias_into_gemm",
    "eliminate_nop_pad",
    "eliminate_nop_dropout",
    "fuse_consecutive_concats",
    "fuse_pad_into_conv",
    "fuse_pad_into_pool",
    "eliminate_nop_reshape",
    "eliminate_nop_flatten",
    "eliminate_shape_op",
    "extract_constant_to_initializer"
]

def generic_optimize(model):
    try:
        return onnxoptimizer.optimize(model, GENERIC_PASSES)
    except Exception as e:
        print(f"Generic optimization failed: {e}")
        return model

def cast_elimination(model: onnx.ModelProto):
    keep_nodes = []
    removed = 0
    for node in model.graph.node:
        if node.op_type == "Cast":
            attr = {a.name: a.i for a in node.attribute}
            to_type = attr.get("to", -1)
            inp_name = node.input[0]
            inp_type = None
            for vi in model.graph.value_info:
                if vi.name == inp_name and vi.type.HasField("tensor_type"):
                    inp_type = vi.type.tensor_type.elem_type
                    break
            if inp_type is not None and inp_type == to_type:
                removed += 1
                continue
        keep_nodes.append(node)
    new_graph = helper.make_graph(keep_nodes, model.graph.name + "_opt", model.graph.input, model.graph.output, model.graph.initializer)
    return helper.make_model(new_graph, ir_version=model.ir_version, opset_imports=model.opset_import), removed

def fp16_surgery(model: onnx.ModelProto):
    model = copy.deepcopy(model)
    conv_count = 0
    for t in model.graph.initializer:
        if t.data_type == TensorProto.FLOAT:
            arr = np.frombuffer(t.raw_data, dtype=np.float32).copy()
            arr_f16 = arr.astype(np.float16)
            t.data_type = TensorProto.FLOAT16
            t.raw_data = arr_f16.tobytes()
            conv_count += 1
    for vi in list(model.graph.value_info) + list(model.graph.input) + list(model.graph.output):
        if vi.type.HasField("tensor_type") and vi.type.tensor_type.elem_type == TensorProto.FLOAT:
            vi.type.tensor_type.elem_type = TensorProto.FLOAT16
    return model, conv_count

def detect_and_replace_symmetry(model):
    # Dynamic graph rewrite for Horizontal, Vertical and Diagonal Symmetry
    model = generic_optimize(model)
    new_nodes = []
    modified = False
    
    # Ensure we have opset 11 to use negative steps in Slice
    opset = model.opset_import[0].version if model.opset_import else 11
    if opset < 11:
        return model
        
    # Pre-create scalar initializers for Slice if needed
    slice_inits = {}
    def get_slice_tensor(name, val):
        if name not in slice_inits:
            slice_inits[name] = helper.make_tensor(name, TensorProto.INT64, [1], [val])
        return name
        
    for node in model.graph.node:
        if node.op_type == "Gather" and len(node.input) >= 2:
            idx_name = node.input[1]
            initializer = next((i for i in model.graph.initializer if i.name == idx_name), None)
            if initializer and initializer.data_type == TensorProto.INT64:
                arr = numpy_helper.to_array(initializer)
                
                # Diagonal Symmetry: check if array represents a transpose mapping
                if len(arr.shape) == 1 and len(arr) > 1 and int(np.sqrt(len(arr)))**2 == len(arr):
                    n = int(np.sqrt(len(arr)))
                    expected_transpose = np.arange(n*n).reshape(n, n).T.flatten()
                    if np.array_equal(arr, expected_transpose):
                        # Replace Gather with Reshape + Transpose + Reshape
                        pass # Could implement, but usually diagonal flips don't use 1D flat gathers in ONNX.
                
                # Horizontal / Vertical Reflection: check if array is reversed indices
                if len(arr.shape) == 1 and len(arr) > 1 and np.array_equal(arr, np.arange(len(arr)-1, -1, -1)):
                    axis = 0
                    for attr in node.attribute:
                        if attr.name == 'axis':
                            axis = attr.i
                            
                    # Replace Gather with Slice with step=-1
                    starts = get_slice_tensor("slice_starts_rev", -1)
                    ends = get_slice_tensor("slice_ends_rev", -len(arr)-1)
                    axes = get_slice_tensor(f"slice_axes_{axis}", axis)
                    steps = get_slice_tensor("slice_step_rev", -1)
                    
                    slice_node = helper.make_node(
                        "Slice",
                        inputs=[node.input[0], starts, ends, axes, steps],
                        outputs=node.output,
                        name=node.name + "_slice_replaced"
                    )
                    new_nodes.append(slice_node)
                    modified = True
                    continue
                    
        new_nodes.append(node)
        
    if modified:
        model.graph.initializer.extend(slice_inits.values())
        model.graph.ClearField("node")
        model.graph.node.extend(new_nodes)
        
    return generic_optimize(model)

def optimize_task(model, fpath):
    model, _ = cast_elimination(model)
    model, _ = fp16_surgery(model)
    
    ops = set(n.op_type for n in model.graph.node)
    # Apply dynamic symmetry replacements
    model = detect_and_replace_symmetry(model)
        
    used_initializers = set()
    for n in model.graph.node:
        used_initializers.update(n.input)
    new_inits = [t for t in model.graph.initializer if t.name in used_initializers]
    model.graph.ClearField("initializer")
    model.graph.initializer.extend(new_inits)
    
    return model

def apply_optimizations():
    onnx_files = sorted(glob.glob(os.path.join(SUBMISSION_DIR, 'task*.onnx')))
    onnx_files = [f for f in onnx_files if not f.endswith('_int8.onnx')]
    print(f"Found {len(onnx_files)} ONNX files to optimize.")
    optimized_count = 0
    for fpath in onnx_files:
        try:
            model = onnx.load(fpath)
            opt_model = optimize_task(model, fpath)
            out_path = os.path.join(optimized_dir, os.path.basename(fpath))
            onnx.save(opt_model, out_path)
            optimized_count += 1
        except Exception as e:
            print(f"Error optimizing {fpath}: {e}")
    print(f"Optimization complete. Saved {optimized_count} files.")

apply_optimizations()


Found 400 ONNX files to optimize.


/var/folders/dt/hwntsksn383f5_5_yt898yrm0000gq/T/ipykernel_23619/2822204721.py:67: RuntimeWarning: overflow encountered in cast
  arr_f16 = arr.astype(np.float16)


Optimization complete. Saved 400 files.
